In [ ]:
from ipyleaflet import Map, Marker, FullScreenControl, ScaleControl, WidgetControl, LegendControl, GeoJSON
import ipyleaflet
import geopandas as gpd
import pandas as pd
import shapely as shp
from shapely.geometry import Polygon, Point, shape
from pyproj import Geod
import ipywidgets as widgets
import osmium as omi
from osmium import FileProcessor as FProc
import os
import io
import json
import re

import numpy as np

def regparse_othertags(row):
    regpat = r'"([^"]+)"=>"([^"]+)"'
    matches = re.findall(regpat, row)
    result = {key: value for key, value in matches}
    return pd.Series(result)


ddm_hydrologic = "./DDM_HydroLogic_flow_cachment.gpkg"
filefull = "./atlarea.osm.pbf"
fileroads = "./atlarea-roads.osm.pbf"
outputfile = "./floodedatl.osm.pbf"


g = Geod(ellps="WGS84")

rds = gpd.read_file(fileroads, layer="lines")
rds = rds.fillna("")
rds = rds.drop("other_tags",axis=1).join(rds["other_tags"].apply(regparse_othertags))
rds = rds.iloc[:,[0,1,2,9,43]]
unwanted = ["footway","path","steps","cycleway","track","corridor",
            "pedestrian","construction","bridleway","raceway","proposed",
            "road","services","disused","busway","rest_area","ramp"]
rds = rds[~rds.highway.isin(unwanted)]

flowpaths = gpd.read_file(ddm_hydrologic, layer="flow_paths").set_crs("EPSG:4269", allow_override=True)
catchments = gpd.read_file(ddm_hydrologic, layer="subcatchments").set_crs("EPSG:4269", allow_override=True)
catchments["area"] = catchments.to_crs("EPSG:6933").area/1000000
flowpaths["len"] = flowpaths.to_crs("EPSG:6933").length/1000
catchments = catchments[catchments["area"]>10].reset_index(names="catchment")
catchments = catchments.iloc[:,[0,6,7]]
flowpaths = flowpaths.iloc[:,[0,6,8,9]]

intersections = gpd.overlay(rds, flowpaths.to_crs("EPSG:4326"), how="intersection", keep_geom_type=False)
intersections = gpd.overlay(intersections, catchments.to_crs("EPSG:4326"), how="intersection")

flowpaths.strahler.value_counts()

In [ ]:
assumptions = {"Bridge immunity":True,"Highway immunity":True,"Trunk immunity":True,"Primary immunity":True,"Secondary immunity":False}
strahlers = {row.catchment:(4,6) for i,row in catchments.iterrows()}

m = Map(center=(33.75,-84.2), zoom=10, scroll_wheel_zoom=False, 
        layout={"height": "900px", "width": "100%"})
m.add(ScaleControl(max_width=200, position='bottomleft'))

strahlist = [widgets.HBox([widgets.Label(value="Catchment"), widgets.IntRangeSlider(
    value=v, min=1, max=flowpaths.strahler.max(), step=1, description=f"{k}", disabled=False)]) 
    for k, v in strahlers.items()]
assumptionlist = [widgets.Checkbox(
    value=v, description=f"{k}",
    disabled=False, indent=False) for k,v in assumptions.items()]
def updateassump(change):
    assumptions[change.owner.description] = change.new
    upcrossings()
def updatestrahl(change):
    strahlers[int(change.owner.description)] = change.new
    active.value = f"{change.owner.description}"
    upcrossings()

[assump.observe(updateassump, names="value") for assump in assumptionlist]
[strahl.children[1].observe(updatestrahl, names="value") for strahl in strahlist]
accordion = widgets.Accordion(children=[widgets.VBox(strahlist), widgets.VBox(assumptionlist)], titles=("Strahler Settings", "Assumptions"))

brdatawidget = WidgetControl(widget=accordion, position="bottomright", layout=widgets.Layout(max_width="50%"))
m.add(brdatawidget)


colormap = {6:"#000000",5:"#17135C",4:"#2d3650",3:"#506b82",2:"#749fb3",1:"#A1A1A1"}
legend = LegendControl(colormap, title="Strahler", position="topleft")
m.add(legend)


riv = gpd.overlay(catchments, flowpaths, how="intersection", keep_geom_type=False)
riv["color"] = riv.strahler.map(colormap)
riv = riv.to_crs("EPSG:4326")
rivjson = json.loads(riv.to_json())
for feature in rivjson["features"]:
    if feature["properties"]["strahler"] <= 4:
        feature["properties"]["style"] = {
            "color": feature["properties"]["color"],
            "weight": min(4, 2+feature["properties"]["strahler"])
        }
    else:
        feature["properties"]["style"] = {
            "color": feature["properties"]["color"],
            "weight": 2*feature["properties"]["strahler"]-2
        }
paths = GeoJSON(data=rivjson)
m.add(paths)

catchjson = json.loads(catchments.to_crs("EPSG:4326").to_json())
areas = GeoJSON(data=catchjson, style={"color": "orange",
        "fillOpacity":0.05,"weight": 6},hover_style={
        "color": "white", 'fillOpacity': 0.2})
def hovinfo(event, feature, properties, id, coordinates):
    hovdata.value = f"Catchment: {properties['catchment']}"
    return 
def areaselect(event, feature, properties, id, coordinates):
    active.value = f"{properties['catchment']}"
    upcrossings()
    return
def viewall(event):
    m1 = intersections.bridge!="yes" if assumptions["Bridge immunity"] else True
    immunerds = []
    if assumptions["Highway immunity"]:
        immunerds.extend(["motorway","motorway_link"])
    if assumptions["Trunk immunity"]:
        immunerds.extend(["trunk","trunk_link"])
    if assumptions["Primary immunity"]:
        immunerds.extend(["primary","primary_link"])
    if assumptions["Secondary immunity"]:
        immunerds.extend(["secondary","secondary_link"])
    mask = False
    for k,v in strahlers.items():
        mask |= ((v[0]<=intersections.strahler)&(intersections.strahler<=v[1])&(intersections.catchment==k))
    paredxing = intersections[(m1)&(~intersections.highway.isin(immunerds))][mask]
    xing.data = json.loads(paredxing.to_json())
hovdata = widgets.Label(value="Catchment: #")
activepre = widgets.Label(value="Active:")
active = widgets.Label(value="0")
displayall = widgets.Button(description="Display All",disabled=False)
displayall.on_click(viewall)
m.add(WidgetControl(widget=widgets.VBox([displayall,widgets.HBox([activepre,active]),hovdata]), position="bottomleft"))
areas.on_hover(hovinfo)
areas.on_click(areaselect)
m.add(areas)


def upcrossings():
    bot,top = strahlers[int(active.value)]
    m1 = intersections.bridge!="yes" if assumptions["Bridge immunity"] else True
    immunerds = []
    if assumptions["Highway immunity"]:
        immunerds.extend(["motorway","motorway_link"])
    if assumptions["Trunk immunity"]:
        immunerds.extend(["trunk","trunk_link"])
    if assumptions["Primary immunity"]:
        immunerds.extend(["primary","primary_link"])
    if assumptions["Secondary immunity"]:
        immunerds.extend(["secondary","secondary_link"])
    paredxing = intersections[(intersections.strahler<=top)&(bot<=intersections.strahler)&(m1)&(~intersections.highway.isin(immunerds))&(intersections.catchment==int(active.value))]
    xing.data = json.loads(paredxing.to_json())
markerstyle = {"radius": 5, "color":"red"}
xing = GeoJSON(data=json.loads(intersections[:0].to_json()),point_style=markerstyle)
m.add(xing)


display(m)

In [ ]:
m1 = intersections.bridge!="yes" if assumptions["Bridge immunity"] else True
immunerds = []
if assumptions["Highway immunity"]:
    immunerds.extend(["motorway","motorway_link"])
if assumptions["Trunk immunity"]:
    immunerds.extend(["trunk","trunk_link"])
if assumptions["Primary immunity"]:
    immunerds.extend(["primary","primary_link"])
if assumptions["Secondary immunity"]:
    immunerds.extend(["secondary","secondary_link"])
mask = False
for k,v in strahlers.items():
    mask |= ((v[0]<=intersections.strahler)&(intersections.strahler<=v[1])&(intersections.catchment==k))
floodpoints = intersections[(m1)&(~intersections.highway.isin(immunerds))][mask]
floodpoints.geometry = floodpoints.to_crs("EPSG:3857").buffer(1).to_crs("EPSG:4326")

rds["geometry"] = rds.geometry.line_merge()
rds = rds.explode().reset_index(drop=True)
rds["geo2"] = rds.geometry
diffedroads = gpd.overlay(rds, floodpoints, how="difference")
diff = diffedroads[diffedroads.geom_type=="LineString"]
diff = diffedroads[diffedroads.geometry==diffedroads.geo2]

todrop = rds.set_index("osm_id").drop(diffedroads.osm_id)
idstodrop = todrop.index
toedit = diffedroads.drop(diff.index)
explodedit = toedit.explode()
# Get rid of id's with short length (<50m)
explodedit = toedit.explode()
explodedit["length"] = explodedit.to_crs("EPSG:3857").geometry.length
b = explodedit.groupby("osm_id")["length"].sum()
shortsplits = b[b<50].index

idstodrop = np.concatenate((idstodrop, shortsplits)).astype(int)
toedit = toedit.set_index("osm_id").drop(shortsplits)
idstoedit = np.concatenate([toedit.index]).astype(int)


fakeid = (2**45)-1
explodict = explodedit.reset_index(drop=True).groupby("osm_id")
editids = set(idstoedit)
dropids = set(idstodrop)

bothfilter = omi.filter.IdFilter(editids|dropids)
roadsfilter = omi.filter.KeyFilter("highway")

with omi.SimpleWriter(outputfile, overwrite=True) as writer:
    ways = FProc(filefull, omi.osm.WAY|omi.osm.NODE)\
            .with_locations()\
            .with_filter(roadsfilter)\
            .with_filter(bothfilter)\
            .with_filter(omi.filter.GeoInterfaceFilter())\
            .handler_for_filtered(writer)
    for obj in ways:
        if obj.is_way():
            if obj.id in dropids:
                pass
            elif obj.id in editids:
                geom = shp.geometry.shape(obj.__geo_interface__["geometry"])
                coords = [(lon, lat) for lon, lat in zip(*geom.xy)]
                clon = [x[0] for x in coords]
                clat = [x[1] for x in coords]
                refs = [nod.ref for nod in obj.nodes]
                for _id, row in explodict.get_group(str(obj.id)).iterrows():
                    newwaynodes = []
                    newcoords = [(lon, lat) for lon, lat in zip(*row.geometry.xy)]
                    for newlon, newlat in newcoords:
                        _, _, distances = g.inv([newlon]*len(clon), [newlat]*len(clat), clon, clat)
                        closest = min(distances)
                        if closest < 0.01:
                            newwaynodes.append(refs[distances.index(closest)])
                        else:
                            newwaynodes.append(fakeid)
                            writer.add_node(omi.osm.mutable.Node(id=fakeid, 
                                location=(newlon, newlat)))
                            fakeid-=1
                            newwaynodes.append(fakeid)
                    writer.add_way(omi.osm.mutable.Way(
                            id=fakeid, nodes=newwaynodes,
                            tags=obj.tags))
                    fakeid-=1
    writer.close()
